# Canonical source document graph
Validated ownership, provenance and compatibility adapters. Legacy source projections remain lossless.

In [ ]:
#| default_exp core.document


In [ ]:
#| export
from __future__ import annotations

from copy import deepcopy
from dataclasses import asdict, dataclass, field, is_dataclass
import hashlib
import json
import math
from pathlib import Path
from typing import Any, Mapping


def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)


def content_hash(value: Any) -> str:
    data = value if isinstance(value, bytes) else canonical_json(value).encode('utf-8')
    return hashlib.sha256(data).hexdigest()


@dataclass(frozen=True)
class Document:
    id: str
    title: str = ''
    corpus_id: str = 'default'
    manual_id: str = ''
    aliases: tuple[str, ...] = ()


@dataclass(frozen=True)
class SourceRevision:
    id: str
    document_id: str
    source_uri: str
    content_sha256: str
    assets: dict[str, Any] = field(default_factory=dict)
    revision_label: str = ''
    parser_manifest: dict[str, Any] = field(default_factory=dict)


def source_revision(document_id: str, source_uri: str, content: bytes, *, assets=None,
                    revision_label='', parser_manifest=None) -> SourceRevision:
    """Parser/code versions are provenance, never the immutable input identity."""
    digest = content_hash(content)
    asset_manifest = deepcopy(assets or {})
    asset_identity = {ref: {key: value for key, value in state.items() if key not in {'path', 'error'}} if isinstance(state, dict) else state for ref, state in asset_manifest.items()}
    identity = content_hash({'document': document_id, 'content': digest, 'assets': asset_identity})
    return SourceRevision(identity, document_id, source_uri, digest, asset_manifest,
                          revision_label, deepcopy(parser_manifest or {}))


@dataclass(frozen=True)
class SourceSpan:
    id: str
    revision_id: str
    source_uri: str
    line_start: int | None = None
    line_end: int | None = None
    byte_start: int | None = None
    byte_end: int | None = None
    page: int | None = None
    bbox: tuple[float, float, float, float] | None = None
    region_id: str | None = None
    cell_id: str | None = None
    asset: str | None = None


@dataclass
class Node:
    id: str
    revision_id: str
    kind: str
    content: str
    parent_id: str | None = None
    children: list[str] = field(default_factory=list)
    semantic_roles: list[str] = field(default_factory=list)
    span_ids: list[str] = field(default_factory=list)
    attributes: dict[str, Any] = field(default_factory=dict)
    aliases: list[str] = field(default_factory=list)


@dataclass(frozen=True)
class Relation:
    kind: str
    source_id: str
    target_id: str
    attributes: dict[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class SummaryArtifact:
    node_id: str
    revision_id: str
    text: str
    status: str = 'present'
    coverage: str = 'coverage_unknown'
    covered_node_ids: tuple[str, ...] = ()
    input_fingerprint: str | None = None
    model_settings: dict[str, Any] = field(default_factory=dict)
    prompt_provenance: dict[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class EmbeddingArtifact:
    node_id: str
    revision_id: str
    representation: str
    rendered_input: str
    input_sha256: str
    encoder_fingerprint: str
    vector_reference: str


@dataclass
class DocumentGraph:
    document: Document
    revision: SourceRevision
    nodes: list[Node]
    spans: list[SourceSpan] = field(default_factory=list)
    relations: list[Relation] = field(default_factory=list)
    summaries: list[SummaryArtifact] = field(default_factory=list)
    embeddings: list[EmbeddingArtifact] = field(default_factory=list)
    provenance: dict[str, Any] = field(default_factory=dict)

    def validate(self) -> DocumentGraph:
        if not self.document.id or self.revision.document_id != self.document.id:
            raise ValueError('Revision must belong to its logical document')
        nodes = {n.id: n for n in self.nodes}
        spans = {s.id: s for s in self.spans}
        if len(nodes) != len(self.nodes) or not nodes:
            raise ValueError('Occurrence IDs must be unique and nonempty')
        if len(spans) != len(self.spans):
            raise ValueError('Source span IDs must be unique')
        roots = [n for n in self.nodes if n.parent_id is None]
        if len(roots) != 1:
            raise ValueError('Ownership requires exactly one root')
        for span in self.spans:
            if span.revision_id != self.revision.id:
                raise ValueError('Span belongs to a different source revision')
            for start, end, minimum in [(span.line_start, span.line_end, 1),
                                         (span.byte_start, span.byte_end, 0)]:
                if (start is None) != (end is None) or (start is not None and (start < minimum or end < start)):
                    raise ValueError('Invalid source interval')
            if span.page is not None and span.page < 1:
                raise ValueError('Invalid page number')
            if span.bbox is not None:
                if span.page is None or len(span.bbox) != 4 or not all(math.isfinite(v) for v in span.bbox):
                    raise ValueError('Physical coordinates require a page and finite box')
                x1, y1, x2, y2 = span.bbox
                if x1 < 0 or y1 < 0 or x2 < x1 or y2 < y1:
                    raise ValueError('Invalid physical coordinates')
        for node in self.nodes:
            if not node.id or node.revision_id != self.revision.id or not node.kind:
                raise ValueError('Invalid node identity or revision')
            if len(node.children) != len(set(node.children)):
                raise ValueError('Duplicate ownership edge')
            if any(s not in spans for s in node.span_ids):
                raise ValueError('Unresolved source span')
            for child in node.children:
                if child not in nodes or nodes[child].parent_id != node.id:
                    raise ValueError('Unresolved or inconsistent child ownership')
            if node.parent_id is not None:
                if node.parent_id not in nodes or node.id not in nodes[node.parent_id].children:
                    raise ValueError('Unresolved or inconsistent parent ownership')
            if node.kind == 'table_cell':
                for key in ('row', 'column', 'row_index', 'column_index'):
                    if key in node.attributes and (not isinstance(node.attributes[key], int) or node.attributes[key] < 0):
                        raise ValueError('Invalid table coordinate')
                for key in ('rowspan', 'colspan'):
                    if key in node.attributes and (not isinstance(node.attributes[key], int) or node.attributes[key] < 1):
                        raise ValueError('Invalid table span')
        seen, visiting = set(), set()
        def visit(identity):
            if identity in visiting:
                raise ValueError('Ownership cycle')
            if identity in seen:
                raise ValueError('Multiple ownership')
            visiting.add(identity)
            for child in nodes[identity].children:
                visit(child)
            visiting.remove(identity)
            seen.add(identity)
        visit(roots[0].id)
        if seen != set(nodes):
            raise ValueError('Disconnected ownership or cycle')
        for edge in self.relations:
            if edge.source_id not in nodes or edge.target_id not in nodes:
                raise ValueError('Unresolved relation')
        for artifact in [*self.summaries, *self.embeddings]:
            if artifact.node_id not in nodes or artifact.revision_id != self.revision.id:
                raise ValueError('Artifact attached to wrong node/revision')
        summary_nodes = [s.node_id for s in self.summaries]
        if len(summary_nodes) != len(set(summary_nodes)):
            raise ValueError('Duplicate summary artifact')
        for summary in self.summaries:
            if not isinstance(summary.text, str) or any(n not in nodes for n in summary.covered_node_ids):
                raise ValueError('Invalid summary text or coverage')
            if summary.status not in {'present', 'missing', 'intentionally_ignored', 'intentionally_skipped'}:
                raise ValueError('Invalid summary status')
        for embedding in self.embeddings:
            if embedding.input_sha256 != content_hash(embedding.rendered_input.encode('utf-8')):
                raise ValueError('Embedding input hash does not match rendered input')
        canonical_json(self.to_dict())
        return self

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, value: Mapping[str, Any]) -> DocumentGraph:
        value = deepcopy(dict(value))
        result = cls(Document(**value['document']), SourceRevision(**value['revision']),
                     [Node(**v) for v in value['nodes']],
                     [SourceSpan(**v) for v in value.get('spans', [])],
                     [Relation(**v) for v in value.get('relations', [])],
                     [SummaryArtifact(**v) for v in value.get('summaries', [])],
                     [EmbeddingArtifact(**v) for v in value.get('embeddings', [])],
                     value.get('provenance', {}))
        return result.validate()


def _source_text(value: Any) -> str:
    return value if isinstance(value, str) else canonical_json(value)


def adapt_semantics(tree: Mapping[str, Any], document: Document, revision: SourceRevision) -> DocumentGraph:
    """Import ownership once; retain exact legacy input for a lossless round trip.

    Mirrors may supply a missing summary but cannot silently disagree about kind,
    source content, or an existing summary. New nodes never renumber old aliases.
    """
    original = deepcopy(dict(tree))
    graph = DocumentGraph(document, revision, [], provenance={
        'adapter': 'semantics-v1', 'legacy_artifact': original, 'coverage': 'coverage_unknown'})
    by_alias: dict[tuple[str, str], Node] = {}
    node_lookup: dict[str, Node] = {}
    span_lookup: set[str] = set()
    summary_by_id: dict[str, SummaryArtifact] = {}
    root_id = content_hash([document.id, revision.id, 'document'])
    root = Node(root_id, revision.id, 'document', '', attributes={'title': document.title})
    graph.nodes.append(root)
    node_lookup[root.id] = root

    def add_summary(node, value):
        if value is None or value == '':
            return
        if not isinstance(value, str):
            raise ValueError(f'Non-string summary for {node.id}')
        previous = summary_by_id.get(node.id)
        if previous and previous.text != value:
            graph.provenance.setdefault('alternate_summaries', []).append({'node_id': node.id, 'text': value})
            return
        summary_by_id[node.id] = SummaryArtifact(node.id, revision.id, value)

    def add_span(node, attrs):
        start, end = attrs.get('line_start'), attrs.get('line_end')
        if start is not None and end is not None:
            sid = content_hash([revision.id, revision.source_uri, start, end])
            if sid not in span_lookup:
                graph.spans.append(SourceSpan(sid, revision.id, revision.source_uri, int(start), int(end)))
                span_lookup.add(sid)
            node.span_ids.append(sid)

    def child_objects(value, path):
        if isinstance(value, list):
            for i, child in enumerate(value):
                yield from child_objects(child, f'{path}/{i}')
        elif isinstance(value, dict) and (value.get('type') or value.get('kind')):
            yield value, path

    def order_cells(node):
        if node.kind == 'table_row':
            order = {identity: i for i, identity in enumerate(node.children)}
            node.children.sort(key=lambda identity: (node_lookup[identity].attributes.get('column', node_lookup[identity].attributes.get('column_index', order[identity])), order[identity]))

    def object_node(obj, parent, path):
        kind = obj.get('kind') or obj.get('type') or 'unknown'
        roles = list(obj.get('semantic_roles') or [])
        if kind == 'task':
            kind = 'table'
            roles = list(dict.fromkeys([*roles, 'task']))
        raw_alias = obj.get('id')
        alias = raw_alias if isinstance(raw_alias, str) else (canonical_json({'legacy_id': raw_alias}) if raw_alias is not None else '')
        content = _source_text(obj.get('content', ''))
        alias_key = (type(raw_alias).__name__, canonical_json(raw_alias)) if raw_alias is not None else None
        prior = by_alias.get(alias_key) if alias_key is not None else None
        if prior:
            if path.rsplit('/', 2)[-2] == 'children':
                ancestor = parent
                while ancestor is not None:
                    if prior.id == ancestor.id:
                        raise ValueError('Self ownership or ownership cycle in legacy object children')
                    ancestor = node_lookup.get(ancestor.parent_id)
            if prior.kind != kind or prior.content != content:
                raise ValueError(f'Conflicting object mirrors for {alias}')
            add_summary(prior, obj.get('summary'))
            # A cells/header mirror may contain descendants absent from children.
            for key in ('children', 'cells', 'headers'):
                for child, child_path in child_objects(obj.get(key, []), f'{path}/{key}'):
                    object_node(child, prior, child_path)
            order_cells(prior)
            return prior
        locator = ['legacy', *alias_key] if alias_key is not None else ['path', path]
        identity = content_hash([document.id, revision.id, kind, locator])
        attrs = deepcopy(obj.get('attributes') or {})
        for key, value in obj.items():
            if key not in {'id', 'type', 'kind', 'semantic_roles', 'content', 'summary', 'children', 'cells', 'headers', 'attributes'}:
                attrs.setdefault(key, deepcopy(value))
        node = Node(identity, revision.id, kind, content, parent.id,
                    semantic_roles=roles, attributes=attrs, aliases=[alias] if alias else [])
        parent.children.append(identity)
        graph.nodes.append(node)
        node_lookup[node.id] = node
        if alias_key is not None:
            by_alias[alias_key] = node
        add_span(node, attrs)
        add_summary(node, obj.get('summary'))
        for key in ('children', 'cells', 'headers'):
            for child, child_path in child_objects(obj.get(key, []), f'{path}/{key}'):
                child_node = object_node(child, node, child_path)
                if key == 'headers' and child_node.id != node.id:
                    edge = Relation('header_for', child_node.id, node.id)
                    if edge not in graph.relations:
                        graph.relations.append(edge)
        order_cells(node)
        return node

    def walk(value, parent, path):
        if isinstance(value, list):
            for i, child in enumerate(value):
                walk(child, parent, f'{path}/{i}')
            return
        if not isinstance(value, dict):
            return
        if isinstance(value.get('object'), dict):
            node = object_node(value['object'], parent, path + '/object')
            add_summary(node, value.get('s'))
            # Object subnodes are mirrors; still visit to recover missing records.
            walk(value.get('subnodes', []), node, path + '/subnodes')
            return
        if value.get('type') in {'table', 'task', 'table_row', 'table_cell', 'table_row_group',
                                  'list', 'list_item', 'figure', 'code', 'formula'}:
            object_node(value, parent, path)
            return
        tag = value.get('t', '')
        if tag in {'Section', 'Para', 'Plain', 'Content', 'BlockQuote', 'CodeBlock', 'Math', 'Header'} or 's' in value:
            kind = {'Section': 'section', 'BlockQuote': 'blockquote', 'CodeBlock': 'code',
                    'Math': 'formula', 'Header': 'heading'}.get(tag, 'paragraph')
            identity = content_hash([document.id, revision.id, kind, path])
            raw_content = value.get('c', '')
            if tag in {'Math', 'CodeBlock', 'Code'} and isinstance(raw_content, list) and len(raw_content) == 2 and isinstance(raw_content[1], str):
                raw_content = raw_content[1]
            node = Node(identity, revision.id, kind, _source_text(raw_content), parent.id,
                        attributes={'legacy_path': path, 'pandoc_type': tag})
            parent.children.append(identity)
            graph.nodes.append(node)
            node_lookup[node.id] = node
            add_summary(node, value.get('s'))
            add_span(node, value)
            # Inline Pandoc leaves are source content, not duplicate prose nodes.
            if 'subnodes' in value:
                walk(value['subnodes'], node, path + '/subnodes')
            elif tag in {'Section', 'BlockQuote', 'Content'}:
                walk(value.get('c', []), node, path + '/c')
            return
        for key in ('blocks', 'subnodes', 'objects'):
            if key in value:
                walk(value[key], parent, path + '/' + key)

    add_summary(root, tree.get('summary'))
    walk(tree, root, '')
    graph.summaries = list(summary_by_id.values())
    return graph.validate()


def export_legacy_semantics(graph: DocumentGraph) -> dict[str, Any]:
    """Return the preserved source projection, never invent historical provenance."""
    graph.validate()
    if 'legacy_artifact' not in graph.provenance:
        raise ValueError('Graph has no legacy semantic projection')
    return deepcopy(graph.provenance['legacy_artifact'])


def adapt_layout(result: Any, document: Document, revision: SourceRevision) -> DocumentGraph:
    """Adapt ingestion records, preserving physical spans only when supplied."""
    records = result.records if hasattr(result, 'records') else result
    region_lookup = result.document.region_map if hasattr(result, 'document') and hasattr(result.document, 'region_map') else {}
    graph = DocumentGraph(document, revision, [], provenance={'adapter': 'layout-v1'})
    root = Node(content_hash([document.id, revision.id, 'document']), revision.id, 'document', '')
    graph.nodes.append(root)
    mapping = {}
    payloads = []
    for record in records:
        payload = asdict(record) if is_dataclass(record) else deepcopy(dict(record))
        alias = str(payload.get('chunk_id') or payload.get('id'))
        if alias in mapping:
            raise ValueError(f'Duplicate layout occurrence {alias}')
        identity = content_hash([document.id, revision.id, 'layout', alias])
        mapping[alias] = identity
        payloads.append((identity, alias, payload))
    for identity, alias, payload in payloads:
        parent = mapping.get(str(payload.get('parent_id')), root.id)
        if parent == identity:
            parent = root.id
        layout_kind = payload.get('content_type', 'region')
        kind = {'prose': 'paragraph', 'table_rows': 'table_row_group'}.get(layout_kind, layout_kind)
        node = Node(identity, revision.id, kind,
                    str(payload.get('exact_text', payload.get('content', payload.get('retrieval_text', '')))), parent,
                    attributes=payload, aliases=[alias])
        graph.nodes.append(node)
        physical = payload.get('source_regions') or payload.get('regions') or []
        if not physical:
            for region_id in payload.get('region_ids', []):
                region = region_lookup.get(region_id)
                physical.append({'region_id': region_id, **({'page_number': region.page_number, 'bbox': region.bbox.as_tuple()} if region is not None else {})})
        if not physical and payload.get('bbox') is not None:
            physical = [payload]
        for i, region in enumerate(physical):
            if not isinstance(region, dict):
                continue
            page = region.get('page_number', region.get('page'))
            box = region.get('bbox')
            if isinstance(box, dict):
                box = [box[k] for k in ('x1', 'y1', 'x2', 'y2')] if 'x1' in box else None
            sid = content_hash([revision.id, alias, i, region])
            span = SourceSpan(sid, revision.id, revision.source_uri,
                              page=int(page) if page is not None else None,
                              bbox=tuple(box) if page is not None and box is not None else None,
                              region_id=str(region.get('region_id', region.get('region_index', ''))) or None)
            graph.spans.append(span)
            node.span_ids.append(sid)
    nodes = {n.id: n for n in graph.nodes}
    for node in graph.nodes[1:]:
        nodes[node.parent_id].children.append(node.id)
    return graph.validate()


def referenced_asset_fingerprints(file) -> dict[str, Any]:
    """Observe local referenced image bytes without fetching remote resources."""
    from html.parser import HTMLParser
    from urllib.parse import unquote, urlsplit
    from markdown_it import MarkdownIt
    file = Path(file)
    refs = set()
    class Images(HTMLParser):
        def handle_starttag(self, tag, attrs):
            if tag == 'img' and (src := dict(attrs).get('src')):
                refs.add(src)
    def collect(tokens):
        for token in tokens:
            if token.type == 'image' and (src := token.attrGet('src')):
                refs.add(src)
            if token.type in {'html_inline', 'html_block'}:
                Images().feed(token.content)
            if token.children:
                collect(token.children)
    collect(MarkdownIt('commonmark', {'html': True}).parse(file.read_text(encoding='utf-8')))
    observed = {}
    for ref in sorted(refs):
        parsed = urlsplit(ref)
        if parsed.scheme or ref.startswith('//'):
            observed[ref] = dict(state='remote_unverified', uri=ref)
            continue
        path = (file.parent / unquote(parsed.path)).resolve()
        try:
            observed[ref] = dict(state='available', path=str(path), sha256=content_hash(path.read_bytes()))
        except FileNotFoundError:
            observed[ref] = dict(state='missing', path=str(path))
        except OSError as exc:
            observed[ref] = dict(state='unavailable', path=str(path), error=type(exc).__name__)
    return observed
